# TQIP preprocessing example

This notebook demonstrates the recommended step-by-step preprocessing order for notebook users.

## 1) Imports and paths

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt

from src.cleaning_utils import (
    AIS_COLUMNS,
    OHE_RENAME_MAP,
    add_verification_level_columns,
    apply_cohort_exclusions,
    drop_original_encoded_columns,
    harmonize_binary_variables,
    load_tqip_data,
    normalize_temperature_to_celsius,
    one_hot_encode_features,
    process_ais_features,
    summarize_raw_cohort,
    validate_required_columns,
)
from src.plotting_utils import (
    plot_categorical_feature_distributions,
    plot_numeric_feature_distributions,
    plot_patient_count_by_year,
    plot_temperature_distribution,
    plot_withdrawal_lst_by_year,
)

repo_root = Path('..')
input_path = repo_root / 'data' / 'raw' / 'VAP_TQIP_2017-2022.xlsx'
output_csv = repo_root / 'data' / 'processed' / 'vap_ohe.csv'
fig_dir = repo_root / 'figures'
fig_dir.mkdir(parents=True, exist_ok=True)

## 2) Load, validate, and summarize the raw cohort

In [ ]:
vap = load_tqip_data(str(input_path))
validate_required_columns(vap)
summary = summarize_raw_cohort(vap)
print('Total patients:', summary['total_patients'])
summary['year_counts']

## 3) Apply ordered exclusions

In [ ]:
vap, exclusion_steps = apply_cohort_exclusions(vap, adult_age=18, min_riss=16)
for step in exclusion_steps:
    print(step)

## 4) Harmonize binary variables and normalize temperature

In [ ]:
vap, binary_audit = harmonize_binary_variables(vap)
vap, converted_n = normalize_temperature_to_celsius(vap)
print('Converted Fahrenheit rows:', converted_n)

## 5) AIS processing and feature engineering

In [ ]:
vap = process_ais_features(vap, ais_cols=AIS_COLUMNS, ais_thresh=3)
vap = one_hot_encode_features(
    vap,
    columns_to_encode=['SEX', 'HMRRHGCTRLSURGTYPE'],
    rename_map=OHE_RENAME_MAP,
)
vap = add_verification_level_columns(vap)
vap_ohe = drop_original_encoded_columns(
    vap,
    columns=['SEX', 'HMRRHGCTRLSURGTYPE', 'VERIFICATIONLEVEL'],
)
vap_ohe.head()

## 6) Notebook-friendly plotting examples

In [ ]:
num_features = [
    'TOTALVENTDAYS', 'AGEyears', 'HMRRHGCTRLSURGMins', 'HMRRHGCTRLSURGDays',
    'WITHDRAWALLSTMins', 'WITHDRAWALLSTDays', 'TOTALICULOS', 'Hospital_LOS_Hr',
    'Hospital_LOS_Days', 'riss', 'mxaisbr_HeadNeck', 'mxaisbr_Face',
    'mxaisbr_Chest', 'mxaisbr_Abdomen', 'mxaisbr_Extremities', 'SBP', 'PULSERATE',
    'TEMPERATURE', 'RESPIRATORYRATE', 'PULSEOXIMETRY', 'PRBC_4', 'FFP_4', 'PLT_4',
    'WB', 'WB_time_mins'
]

cat_features = [
    'HC_VAPNEUMONIA', 'WHITE', 'BLACK', 'ETHNICITY', 'PACIFICISLANDER', 'AMERICANINDIAN',
    'ASIAN', 'RACEOTHER', 'WITHDRAWALLST', 'INTERFACILITYTRANSFER', 'RESPIRATORYASSISTANCE',
    'SUPPLEMENTALOXYGEN', 'PREHOSPITALCARDIACARREST', 'Mortality', 'CC_CHEMO',
    'CC_CIRRHOSIS', 'CC_COPD', 'CC_CVA', 'CC_DIABETES', 'CC_DISCANCER', 'CC_FUNCTIONAL',
    'CC_CHF', 'CC_RENAL', 'CC_SMOKING', 'ICP_Monitor', 'sex_male', 'sex_female',
    'sex_non-binary', 'surg_none', 'surg_laparotomy', 'surg_thoracotomy', 'surg_sternotomy',
    'surg_extremity', 'surg_neck', 'surg_amputation', 'surg_skin_softtissue',
    'surg_pelvic_packing', 'inj_HeadNeck', 'inj_Face', 'inj_Chest', 'inj_Abdomen',
    'inj_Extremities', 'L1', 'L2', 'L3'
]

for func, name in [
    (lambda d: plot_patient_count_by_year(d), 'patient_count_by_year.png'),
    (lambda d: plot_withdrawal_lst_by_year(d), 'withdrawal_lst_by_year.png'),
    (lambda d: plot_temperature_distribution(d), 'temperature_distribution.png'),
    (lambda d: plot_numeric_feature_distributions(d, num_features), 'numeric_feature_distributions.png'),
    (lambda d: plot_categorical_feature_distributions(d, cat_features), 'categorical_feature_distributions.png'),
]:
    fig, _ = func(vap_ohe)
    fig.savefig(fig_dir / name, dpi=300, bbox_inches='tight')
    plt.close(fig)

## 7) Inspect and save final dataframe

In [ ]:
print(vap_ohe.shape)
output_csv.parent.mkdir(parents=True, exist_ok=True)
vap_ohe.to_csv(output_csv, index=False)
output_csv